In [0]:
from pyspark.sql import functions as F

events = spark.table("events_delta")
features_df = spark.table("silver_user_features")

In [0]:
label_df = events.groupBy("user_id") \
    .agg(
        F.max(
            F.when(F.col("event_type") == "purchase", 1).otherwise(0)
        ).alias("purchased")
    )

In [0]:
label_df.display()

In [0]:
training_data = features_df.join(label_df, "user_id", "inner")

In [0]:
training_data.display()

In [0]:
training_data.groupBy("user_id").count().filter("count > 1").display()

In [0]:
from pyspark.ml.feature import VectorAssembler

feature_cols = ["total_events", "purchases", "total_spent", "avg_price"]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

ml_ready_df = assembler.transform(training_data) \
    .select("features", F.col("purchased").alias("label"))

In [0]:
ml_ready_df.display()

In [0]:
train, test = ml_ready_df.randomSplit([0.8, 0.2], seed=42)

In [0]:
print("Train count:", train.count())
print("Test count:", test.count())

In [0]:
training_data.groupBy("purchased").count().display()


In [0]:
train.groupBy("label").count().display()

In [0]:
test.groupBy("label").count().display()

In [0]:
train.write.format("delta").mode("overwrite").saveAsTable("train_dataset")
test.write.format("delta").mode("overwrite").saveAsTable("test_dataset")